In [1]:
df = spark.table("hospital_silver")

StatementMeta(, 2a64eb0c-6745-4b2a-9629-5bc6f1540d37, 3, Finished, Available, Finished, False)

In [2]:
df.printSchema()

StatementMeta(, 2a64eb0c-6745-4b2a-9629-5bc6f1540d37, 4, Finished, Available, Finished, False)

root
 |-- Facility_ID: double (nullable = true)
 |-- Facility_Name: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- City_Town: string (nullable = true)
 |-- State: string (nullable = true)
 |-- ZIP_Code: integer (nullable = true)
 |-- County_Parish: string (nullable = true)
 |-- Telephone_Number: string (nullable = true)
 |-- Hospital_Type: string (nullable = true)
 |-- Hospital_Ownership: string (nullable = true)
 |-- Emergency_Services: string (nullable = true)
 |-- Meets_criteria_for_birthing_friendly_designation: string (nullable = true)
 |-- Hospital_overall_rating: string (nullable = true)
 |-- Hospital_overall_rating_footnote: string (nullable = true)
 |-- MORT_Group_Measure_Count: string (nullable = true)
 |-- Count_of_Facility_MORT_Measures: string (nullable = true)
 |-- Count_of_MORT_Measures_Better: string (nullable = true)
 |-- Count_of_MORT_Measures_No_Different: string (nullable = true)
 |-- Count_of_MORT_Measures_Worse: string (nullable = true)
 |-- 

In [3]:
df = spark.table("hospital_silver")

StatementMeta(, 2a64eb0c-6745-4b2a-9629-5bc6f1540d37, 5, Finished, Available, Finished, False)

In [4]:
from pyspark.sql.functions import count, avg

gold_state_summary = (
    df.groupBy("State")
      .agg(
          count("*").alias("Total_Hospitals"),
          avg("Hospital_overall_rating").alias("Average_Rating")
      )
)

gold_state_summary.show()

StatementMeta(, 2a64eb0c-6745-4b2a-9629-5bc6f1540d37, 7, Finished, Available, Finished, False)

+-----+---------------+------------------+
|State|Total_Hospitals|    Average_Rating|
+-----+---------------+------------------+
|   SC|             66|3.4285714285714284|
|   AZ|            107| 3.109090909090909|
|   LA|            161|2.8135593220338984|
|   MN|            136|3.7719298245614037|
|   NJ|             79|3.1475409836065573|
|   DC|             10| 2.857142857142857|
|   OR|             62|              3.25|
|   VA|             96|3.6842105263157894|
|   RI|             13|3.5454545454545454|
|   WY|             30|             3.375|
|   KY|            103|2.7611940298507465|
|   NH|             28|3.1739130434782608|
|   MI|            148|3.0851063829787235|
|   NV|             46|               3.0|
|   WI|            141|3.7790697674418605|
|   ID|             48|3.5454545454545454|
|   CA|            378|               3.1|
|   NE|             93|               3.5|
|   CT|             36| 3.423076923076923|
|   MT|             63|              3.25|
+-----+----

In [5]:
gold_state_summary.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_state_summary")

StatementMeta(, 2a64eb0c-6745-4b2a-9629-5bc6f1540d37, 9, Finished, Available, Finished, False)

In [6]:
from pyspark.sql.functions import count

gold_emergency_services = (
    df.groupBy("Emergency_Services")
      .agg(count("*").alias("Hospital_Count"))
)

gold_emergency_services.show()

StatementMeta(, 2a64eb0c-6745-4b2a-9629-5bc6f1540d37, 11, Finished, Available, Finished, False)

+------------------+--------------+
|Emergency_Services|Hospital_Count|
+------------------+--------------+
|                No|           934|
|               Yes|          4498|
+------------------+--------------+



In [7]:
gold_emergency_services.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_emergency_services")

StatementMeta(, 2a64eb0c-6745-4b2a-9629-5bc6f1540d37, 13, Finished, Available, Finished, False)

In [8]:
gold_ownership = (
    df.groupBy("Hospital_Ownership")
      .agg(count("*").alias("Hospital_Count"))
)

gold_ownership.show()

StatementMeta(, 2a64eb0c-6745-4b2a-9629-5bc6f1540d37, 15, Finished, Available, Finished, False)

+--------------------+--------------+
|  Hospital_Ownership|Hospital_Count|
+--------------------+--------------+
|           Physician|            81|
|         Proprietary|          1067|
|Government - Federal|            43|
|Voluntary non-pro...|          2316|
|              Tribal|            16|
|  Government - State|           209|
|Voluntary non-pro...|           355|
|Veterans Health A...|           132|
|Voluntary non-pro...|           268|
|Government - Hosp...|           516|
|Department of Def...|            32|
|  Government - Local|           397|
+--------------------+--------------+



In [9]:
gold_ownership.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_hospital_ownership")

StatementMeta(, 2a64eb0c-6745-4b2a-9629-5bc6f1540d37, 17, Finished, Available, Finished, False)

In [10]:
gold_hospital_type = (
    df.groupBy("Hospital_Type")
      .agg(count("*").alias("Hospital_Count"))
)

gold_hospital_type.show()

StatementMeta(, 2a64eb0c-6745-4b2a-9629-5bc6f1540d37, 19, Finished, Available, Finished, False)

+--------------------+--------------+
|       Hospital_Type|Hospital_Count|
+--------------------+--------------+
|Acute Care - Vete...|           132|
|Acute Care Hospitals|          3115|
|Rural Emergency H...|            41|
|         Psychiatric|           635|
|           Long-term|             5|
|Acute Care - Depa...|            32|
|Critical Access H...|          1378|
|           Childrens|            94|
+--------------------+--------------+



In [11]:
gold_hospital_type.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_hospital_type")

StatementMeta(, 2a64eb0c-6745-4b2a-9629-5bc6f1540d37, 21, Finished, Available, Finished, False)

In [12]:
gold_top_hospitals = (
    df.filter(df.Hospital_overall_rating == "5")
      .select(
          "Facility_Name",
          "State",
          "City_Town",
          "Hospital_overall_rating"
      )
)

gold_top_hospitals.show(20, truncate=False)

StatementMeta(, 2a64eb0c-6745-4b2a-9629-5bc6f1540d37, 22, Finished, Available, Finished, False)

+--------------------------------------------------+-----+---------------+-----------------------+
|Facility_Name                                     |State|City_Town      |Hospital_overall_rating|
+--------------------------------------------------+-----+---------------+-----------------------+
|ASCENSION SE WISCONSIN HOSPITAL                   |WI   |MILWAUKEE      |5                      |
|ST ALEXIUS MEDICAL CENTER                         |IL   |HOFFMAN ESTATES|5                      |
|ST JOHN OWASSO                                    |OK   |OWASSO         |5                      |
|PENINSULA MEDICAL CENTER                          |CA   |BURLINGAME     |5                      |
|MEDICAL CENTER OF THE ROCKIES                     |CO   |LOVELAND       |5                      |
|HCA FLORIDA SOUTH TAMPA HOSPITAL                  |FL   |TAMPA          |5                      |
|VAN WERT COUNTY HOSPITAL                          |OH   |VAN WERT       |5                      |
|ST ELIZAB

In [13]:
gold_top_hospitals.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_top_hospitals")

StatementMeta(, 2a64eb0c-6745-4b2a-9629-5bc6f1540d37, 23, Finished, Available, Finished, False)

In [14]:
gold_birth_friendly = (
    df.filter(df.Meets_criteria_for_birthing_friendly_designation == "Yes")
      .select(
          "Facility_Name",
          "State",
          "City_Town",
          "Hospital_Ownership"
      )
)

gold_birth_friendly.show()

StatementMeta(, 2a64eb0c-6745-4b2a-9629-5bc6f1540d37, 25, Finished, Available, Finished, False)

+-------------+-----+---------+------------------+
|Facility_Name|State|City_Town|Hospital_Ownership|
+-------------+-----+---------+------------------+
+-------------+-----+---------+------------------+



In [15]:
df.select("Meets_criteria_for_birthing_friendly_designation").distinct().show(50, truncate=False)

StatementMeta(, 2a64eb0c-6745-4b2a-9629-5bc6f1540d37, 27, Finished, Available, Finished, False)

+------------------------------------------------+
|Meets_criteria_for_birthing_friendly_designation|
+------------------------------------------------+
|Y                                               |
|Unknown                                         |
+------------------------------------------------+



In [16]:
gold_birth_friendly = (
    df.filter(df.Meets_criteria_for_birthing_friendly_designation == "Y")
      .select(
          "Facility_Name",
          "State",
          "City_Town",
          "Hospital_Ownership"
      )
)

gold_birth_friendly.show()

StatementMeta(, 2a64eb0c-6745-4b2a-9629-5bc6f1540d37, 29, Finished, Available, Finished, False)

+--------------------+-----+-----------------+--------------------+
|       Facility_Name|State|        City_Town|  Hospital_Ownership|
+--------------------+-----+-----------------+--------------------+
|SACRED HEART HOSP...|   FL|        PENSACOLA|Voluntary non-pro...|
|ASCENSION NE WISC...|   WI|         APPLETON|Voluntary non-pro...|
|ASCENSION COLUMBI...|   WI|        MILWAUKEE|Voluntary non-pro...|
|ST VINCENT'S BIRM...|   AL|       BIRMINGHAM|Voluntary non-pro...|
|PRESENCE SAINT JO...|   IL|          CHICAGO|Voluntary non-pro...|
|ASCENSION PROVIDE...|   MI|        ROCHESTER|Voluntary non-pro...|
|PROVIDENCE MEDICA...|   NE|            WAYNE|Voluntary non-pro...|
|ASCENSION ST VINC...|   FL|       MIDDLEBURG|Voluntary non-pro...|
|PRESENCE SAINTS M...|   IL|          CHICAGO|Voluntary non-pro...|
|ASPIRUS IRONWOOD ...|   MI|         IRONWOOD|Voluntary non-pro...|
|KITTITAS VALLEY C...|   WA|       ELLENSBURG|Government - Hosp...|
|ASCENSION  SETON ...|   TX|             KYLE|Vo

In [17]:
gold_birth_friendly.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_birth_friendly")

StatementMeta(, 2a64eb0c-6745-4b2a-9629-5bc6f1540d37, 31, Finished, Available, Finished, False)

In [18]:
spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, 2a64eb0c-6745-4b2a-9629-5bc6f1540d37, 32, Finished, Available, Finished, False)

+--------------------------------------+--------------------------------+-----------+
|namespace                             |tableName                       |isTemporary|
+--------------------------------------+--------------------------------+-----------+
|`My workspace`.HealthcareLakehouse.dbo|bronze_hospital_information     |false      |
|`My workspace`.HealthcareLakehouse.dbo|bronze_provider_services        |false      |
|`My workspace`.HealthcareLakehouse.dbo|bronze_unplanned_hospital_visits|false      |
|`My workspace`.HealthcareLakehouse.dbo|gold_birth_friendly             |false      |
|`My workspace`.HealthcareLakehouse.dbo|gold_emergency_services         |false      |
|`My workspace`.HealthcareLakehouse.dbo|gold_hospital_ownership         |false      |
|`My workspace`.HealthcareLakehouse.dbo|gold_hospital_type              |false      |
|`My workspace`.HealthcareLakehouse.dbo|gold_state_summary              |false      |
|`My workspace`.HealthcareLakehouse.dbo|gold_top_hospi

In [19]:
spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, 2a64eb0c-6745-4b2a-9629-5bc6f1540d37, 33, Finished, Available, Finished, False)

+--------------------------------------+--------------------------------+-----------+
|namespace                             |tableName                       |isTemporary|
+--------------------------------------+--------------------------------+-----------+
|`My workspace`.HealthcareLakehouse.dbo|bronze_hospital_information     |false      |
|`My workspace`.HealthcareLakehouse.dbo|bronze_provider_services        |false      |
|`My workspace`.HealthcareLakehouse.dbo|bronze_unplanned_hospital_visits|false      |
|`My workspace`.HealthcareLakehouse.dbo|gold_birth_friendly             |false      |
|`My workspace`.HealthcareLakehouse.dbo|gold_emergency_services         |false      |
|`My workspace`.HealthcareLakehouse.dbo|gold_hospital_ownership         |false      |
|`My workspace`.HealthcareLakehouse.dbo|gold_hospital_type              |false      |
|`My workspace`.HealthcareLakehouse.dbo|gold_state_summary              |false      |
|`My workspace`.HealthcareLakehouse.dbo|gold_top_hospi